In [1]:
!pip install transformers seqeval evaluate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e88557f52271dd6dc153601be03a97a677ccb403d805f9545a35ac9646858165
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Data Loading

In [2]:
from datasets import load_dataset
dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/266k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/930 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [3]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

Data Pre-Processing

In [4]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
inputs=tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)

###[SEP] é um token especial que indica o fim de uma sequência de texto. Ele é usado para separar diferentes partes do texto, como frases ou parágrafos, e também pode ser usado para indicar o final de uma entrada em tarefas de processamento de linguagem natural, como classificação ou geração de texto. O token [SEP] é importante para ajudar os modelos de linguagem a entenderem a estrutura do texto e a processarem as informações de maneira adequada.
###[CLS] é um token especial que é adicionado no início de uma sequência de texto em tarefas de processamento de linguagem natural, como classificação ou geração de texto. Ele é usado para indicar o início da entrada e é importante para ajudar os modelos de linguagem a entenderem a estrutura do texto e a processarem as informações de maneira adequada. O token [CLS] é frequentemente usado em conjunto com o token [SEP] para separar diferentes partes do texto e indicar o início e o fim da entrada.

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [7]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

In [8]:
tokens = ["as", "aulas", "de", "plneb", "são","interessantes", "!"]
inputs = tokenizer(tokens, is_split_into_words=True) ##aviso de que os tokens já estão separados, para evitar que o tokenizer tente dividir as palavras novamente.

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'de', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [9]:
len(tokens), len(new_tokens)

(7, 11)

In [10]:
inputs.word_ids() ##mapeamento direto entre os tokens e as palavras originais. Cada token é associado a um índice que indica a qual palavra original ele pertence. Se um token for parte de uma palavra original, ele terá o mesmo índice que essa palavra. Se um token for um token especial, como [CLS] ou [SEP], ele terá um índice diferente. O método word_ids() é útil para alinhar as etiquetas de uma tarefa de processamento de linguagem natural com os tokens gerados pelo tokenizer, garantindo que as etiquetas sejam atribuídas corretamente aos tokens correspondentes.

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, None]

In [21]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)  # código que o BERT usa para ignorar tokens especiais
        elif word_id != previous_word:
            new_labels.append(labels[word_id])  # etiqueta da palavra original
        else:
            new_labels.append(-100)  # código para ignorar sub-tokens
        previous_word = word_id

    return new_labels


def tokenize_dataset(dataset):
    res=[]
    for row in dataset:
        inputs=tokenizer(row["tokens"], is_split_into_words=True, truncation=True, max_length=512) ##vai me dar os meus inputs, mas as labels estao mal entao temos de aplicar a função align_labels_with_tokens para corrigir as labels
        new_labels=align_labels_with_tokens(inputs.word_ids(), row["ner_tags"]) ##vai me dar as minhas labels corrigidas, alinhadas com os tokens gerados pelo tokenizer
        inputs["labels"]=new_labels ##adiciona as labels corrigidas aos meus inputs
        res.append(inputs) ##adiciona os meus inputs corrigidos a uma lista de resultados
    return res


train_dataset = tokenize_dataset(dataset_raw["train"])
test_dataset = tokenize_dataset(dataset_raw["test"])

len(train_dataset), len(test_dataset)



(3716, 930)

In [22]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


Model Training

In [23]:
from transformers import AutoModelForTokenClassification
model=AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [24]:
label_list = dataset_raw["train"].features["ner_tags"].feature.names
label_list

['O',
 'B-Data',
 'I-Data',
 'B-Local',
 'I-Local',
 'B-Organizacao',
 'I-Organizacao',
 'B-Pessoa',
 'I-Pessoa',
 'B-Profissao',
 'I-Profissao']

In [25]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Evaluate

In [26]:
import evaluate

seqeval = evaluate.load("seqeval")

In [27]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Train

In [28]:
id2label = {
    0: "O",
    1: "B-Data",
    2: "I-Data",
    3: "B-Local",
    4: "I-Local",
    5: "B-Organizacao",
    6: "I-Organizacao",
    7: "B-Pessoa",
    8: "I-Pessoa",
    9: "B-Profissao",
    10: "I-Profissao"
}

label2id = {
    "O": 0,
    "B-Data": 1,
    "I-Data": 2,
    "B-Local": 3,
    "I-Local": 4,
    "B-Organizacao": 5,
    "I-Organizacao": 6,
    "B-Pessoa": 7,
    "I-Pessoa": 8,
    "B-Profissao": 9,
    "I-Profissao": 10
}

In [29]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [30]:
training_args = TrainingArguments(
    output_dir="meu_modelo_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.074905,0.931448,0.959164,0.945103,0.981610
2,No log,0.067866,0.943928,0.964546,0.954126,0.983799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=466, training_loss=0.13716017432478875, metrics={'train_runtime': 216.2528, 'train_samples_per_second': 34.367, 'train_steps_per_second': 2.155, 'total_flos': 436227558643800.0, 'train_loss': 0.13716017432478875, 'epoch': 2.0})

Inference

In [31]:
texto_noticia="""A engenheira de software Laura Ferreira foi ontem convidada pela Fundação Calouste Gulbenkian para apresentar um novo projeto de inteligência artificial.
O evento principal ocorrerá no dia 15 de novembro na cidade de Lisboa e contará com várias personalidades do setor tecnológico.
Segundo o diretor da organização, João Costa, a colaboração de investigadores e programadores é fundamental para colocar Portugal na vanguarda da inovação europeia."""

In [32]:
from transformers import pipeline

classifier = pipeline("ner",model=model, tokenizer=tokenizer, aggregation_strategy="first")
classifier(texto_noticia)


[{'entity_group': 'Profissao',
  'score': np.float32(0.6900101),
  'word': 'engenheira',
  'start': 2,
  'end': 12},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.98079383),
  'word': 'Laura Ferreira',
  'start': 25,
  'end': 39},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.6850622),
  'word': 'Fundação Calouste Gulbenkian',
  'start': 65,
  'end': 93},
 {'entity_group': 'Data',
  'score': np.float32(0.796093),
  'word': '15 de novembro',
  'start': 190,
  'end': 204},
 {'entity_group': 'Local',
  'score': np.float32(0.95479256),
  'word': 'Lisboa',
  'start': 218,
  'end': 224},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.9743488),
  'word': 'João Costa',
  'start': 318,
  'end': 328},
 {'entity_group': 'Local',
  'score': np.float32(0.67997795),
  'word': 'Portugal',
  'start': 405,
  'end': 413}]